In [1]:
import numpy as np
from scripts.utilities import *
from scripts.features import *
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn import metrics

In [2]:
consumer_df, account_df, transaction_df = get_data()
transaction_df.amount = transaction_df.amount.apply(abs)

Data successfully loaded and processed.


In [3]:
# c_df = consumer_df.dropna(subset='DQ_TARGET')
# a_df = account_df[account_df.prism_consumer_id.isin(c_df.prism_consumer_id)]
# t_df = transaction_df[transaction_df.prism_consumer_id.isin(c_df.prism_consumer_id)]

In [4]:
c_df = consumer_df
a_df = account_df
t_df = transaction_df

## Account balance overtime:

- Balance recorded at the time account_df was made:

In [5]:
balance = a_df.groupby(['prism_consumer_id']).agg({'balance_date':'max', 'balance':'sum'})
# display(balance)
acct_balance = balance['balance']
# acct_balance

- Current balance:

In [6]:
t = t_df.copy()

In [7]:
t['balance_date'] = balance['balance_date']
t['amount'] = np.where(t['credit_or_debit'] == 'DEBIT', -t['amount'], t['amount'])
t['is_before_balance_date'] = np.where(t['posted_date'] < t['balance_date'], True, False)
update_balance = t[t['is_before_balance_date'] == False]
update_balance = update_balance.groupby(['prism_consumer_id'])['amount'].sum()

In [8]:
current_balance = c_df[['prism_consumer_id']]

current_balance['balance'] = current_balance['prism_consumer_id'].map(acct_balance).fillna(0)
current_balance['update'] = current_balance['prism_consumer_id'].map(update_balance).fillna(0)
current_balance['current_balance'] = current_balance['balance'] + current_balance['update']
current_balance = current_balance.current_balance
# current_balance

C:\Users\bdion\AppData\Local\Temp\ipykernel_2628\921156377.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_balance['balance'] = current_balance['prism_consumer_id'].map(acct_balance).fillna(0)
C:\Users\bdion\AppData\Local\Temp\ipykernel_2628\921156377.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  current_balance['update'] = current_balance['prism_consumer_id'].map(update_balance).fillna(0)


## Spending overtime:

In [9]:
inflows = t_df[t_df.credit_or_debit == 'CREDIT']
outflows = t_df[t_df.credit_or_debit == 'DEBIT']

# display(inflows, outflows)
inflows.shape, outflows.shape

((1104963, 6), (5302358, 6))

In [10]:
avg_spending = outflows.groupby('prism_consumer_id')['amount'].mean()
# avg_spending

In [11]:
outflows['year'] = outflows['posted_date'].dt.year
outflows['month'] = outflows['posted_date'].dt.month
outflows['week'] = outflows['posted_date'].dt.isocalendar().week
monthly_totals = outflows.groupby(['prism_consumer_id', 'year', 'month'])['amount'].sum().groupby('prism_consumer_id').mean()
weekly_totals  = outflows.groupby(['prism_consumer_id', 'year', 'week'])['amount'].sum().groupby('prism_consumer_id').mean()
yearly_totals  = outflows.groupby(['prism_consumer_id', 'year', 'year'])['amount'].sum().groupby('prism_consumer_id').mean()

# display(outflows)

C:\Users\bdion\AppData\Local\Temp\ipykernel_2628\822019574.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outflows['year'] = outflows['posted_date'].dt.year
C:\Users\bdion\AppData\Local\Temp\ipykernel_2628\822019574.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outflows['month'] = outflows['posted_date'].dt.month
C:\Users\bdion\AppData\Local\Temp\ipykernel_2628\822019574.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,co

- Spending within 3 weeks, 6 months, etc.

In [12]:
outflows['posted_date'] = pd.to_datetime(outflows['posted_date'])
spending_over_time = outflows.sort_values(['prism_consumer_id', 'posted_date'])

C:\Users\bdion\AppData\Local\Temp\ipykernel_2628\1537017176.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outflows['posted_date'] = pd.to_datetime(outflows['posted_date'])


In [13]:
initial_dates = outflows.groupby('prism_consumer_id')['posted_date'].min()

In [14]:
spending_over_time = spending_over_time.merge(initial_dates, on='prism_consumer_id', how='left', suffixes=('', '_initial'))
spending_over_time = spending_over_time.rename(columns={'posted_date_initial': 'initial_date'})

In [15]:
spending_over_time['days_between'] = spending_over_time['posted_date'] - spending_over_time['initial_date']

spending_over_time['months_between'] = (
    (spending_over_time['posted_date'].dt.year - spending_over_time['initial_date'].dt.year) * 12 +
    (spending_over_time['posted_date'].dt.month - spending_over_time['initial_date'].dt.month)
).abs()

In [16]:
for weeks in range(7, 53, 7):    
    spending_over_time[f'first_{weeks}_weeks'] = spending_over_time['days_between'].astype('int64') <= weeks

for months in range(3, 13, 3):    
    spending_over_time[f'first_{months}_months'] = spending_over_time['months_between'] <= months

# spending_over_time

In [17]:
month_aggs = c_df[['prism_consumer_id']].drop_duplicates().reset_index(drop=True)

for months in range(3, 13, 3):  
    months_df = spending_over_time[spending_over_time[f'first_{months}_months']]
    
    agg_df = (
        months_df
        .groupby('prism_consumer_id')
        .agg(amount_sum=('amount', 'sum'), amount_std=('amount', 'std'), amount_mean=('amount', 'mean'))
        .reset_index()
    )

    month_aggs = month_aggs.merge(
        agg_df, on='prism_consumer_id', how='left', suffixes=('', f'_first_{months}_months')
    )

month_aggs

,prism_consumer_id,amount_sum,amount_std,amount_mean,amount_sum_first_6_months,amount_std_first_6_months,amount_mean_first_6_months,amount_sum_first_9_months,amount_std_first_9_months,amount_mean_first_9_months,amount_sum_first_12_months,amount_std_first_12_months,amount_mean_first_12_months
0,0,8797.15,73.835391,42.704612,14908.41,70.470938,40.293000,14908.41,70.470938,40.293000,14908.41,70.470938,40.293000
1,1,10605.49,137.990627,84.843920,23098.37,172.961059,95.055021,23098.37,172.961059,95.055021,23098.37,172.961059,95.055021
2,2,11040.12,110.238192,51.589346,22334.58,211.458378,60.857166,22334.58,211.458378,60.857166,22334.58,211.458378,60.857166
3,3,8461.56,105.087958,76.230270,19846.01,276.834340,90.209136,19846.01,276.834340,90.209136,19846.01,276.834340,90.209136
4,4,4888.81,92.358957,68.856479,6288.24,92.880270,62.882400,8849.66,81.981091,55.658239,17509.71,116.561814,65.825977
...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,14995,1557.98,62.141403,48.686875,12098.33,70.203944,48.200518,14780.41,79.511705,53.746945,14780.41,79.511705,53.746945
14996,14996,26695.04,758.492569,360.743784,37659.84,518.078292,196.145000,60990.82,855.201544,249.962377,60990.82,855.201544,249.962377
14997,14997,11595.09,77.646058,64.777039,39763.99,278.952767,114.593631,43695.82,265.080091,110.065038,43695.82,265.080091,110.065038
14998,14998,117251.34,1763.272029,179.833344,148891.70,1319.736513,126.608588,170415.12,1228.889588,123.578767,170415.12,1228.889588,123.578767


- Visualization for balance changes over time:

In [18]:
balance_changes = t[t['is_before_balance_date'] == False].sort_values(['prism_consumer_id', 'posted_date'])
# balance_changes

In [19]:
balance_changes['amount'] = pd.to_numeric(balance_changes['amount'], errors='coerce')

In [20]:
initial_balance = acct_balance.to_dict()

In [21]:
from collections import defaultdict

changes = defaultdict(list)

for id, amount in balance_changes[['prism_consumer_id', 'amount']].values:
    changes[id].append(amount)

In [22]:
balance_change = defaultdict(list)
for i, v in changes.items():  
    if i in acct_balance.keys():
        v.insert(0, acct_balance[i])  
    updated_balance = np.cumsum(v)
    balance_change[i] = updated_balance

print(len(balance_change))

14492


## Getting stats for balance changes overtime:

In [23]:
balance_change = pd.DataFrame(
    [(k, v) for k, lst in balance_change.items() for v in lst], 
    columns=['prism_consumer_id', 'balance_changes']
)
# balance_change

In [24]:
stats_changes = balance_change.groupby(['prism_consumer_id']).agg({'balance_changes': ['mean', 'std']})
# stats_changes

## Required features:

In [25]:
feats = t_df.groupby(['prism_consumer_id', 'category']).agg({'amount': ['count', 'sum', 'std', 'mean', 'median']})
feats = feats.unstack(level=1)
feats.columns = ['_'.join(col).strip() for col in feats.columns.values]
feats = feats.fillna(0)
# feats = feats.reset_index()
# feats.head()

- Putting everything together:

In [26]:
result = c_df[['prism_consumer_id', 'DQ_TARGET']].drop_duplicates().reset_index(drop=True)

result['current_balance'] = result['prism_consumer_id'].map(current_balance)
result['balance_mean'] = result['prism_consumer_id'].map(stats_changes[('balance_changes', 'mean')])
result['balance_std'] = result['prism_consumer_id'].map(stats_changes[('balance_changes', 'std')])
result['avg_spending'] = result['prism_consumer_id'].map(avg_spending).fillna(0)
result['avg_monthly_outflow'] = result['prism_consumer_id'].map(monthly_totals).fillna(0)
result['avg_weekly_outflow'] = result['prism_consumer_id'].map(weekly_totals).fillna(0)
result['avg_yearly_outflow'] = result['prism_consumer_id'].map(yearly_totals).fillna(0)

mapped_feats = pd.DataFrame({col: result['prism_consumer_id'].map(feats[col]) for col in feats.columns})
outflow_feats = pd.DataFrame({col: result['prism_consumer_id'].map(month_aggs[col]) for col in month_aggs.columns})
result = pd.concat([result, mapped_feats, outflow_feats], axis=1)
# result

In [27]:
# result.isna().sum()

In [28]:
result = result.loc[:,~result.columns.duplicated()].copy()
result

,prism_consumer_id,DQ_TARGET,current_balance,balance_mean,balance_std,avg_spending,avg_monthly_outflow,avg_weekly_outflow,avg_yearly_outflow,amount_count_ACCOUNT_FEES,...,amount_mean,amount_sum_first_6_months,amount_std_first_6_months,amount_mean_first_6_months,amount_sum_first_9_months,amount_std_first_9_months,amount_mean_first_9_months,amount_sum_first_12_months,amount_std_first_12_months,amount_mean_first_12_months
0,0,0.0,NaN,334.377359,986.039607,40.293000,2129.772857,573.400385,14908.410,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0.0,NaN,5008.010698,1196.388690,95.055021,3299.767143,855.495185,23098.370,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,0.0,NaN,5022.325479,2504.559496,60.857166,3190.654286,797.663571,11167.290,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,0.0,NaN,7278.493493,1939.603669,90.209136,2835.144286,708.786071,9923.005,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,0.0,NaN,-674.058730,659.354952,65.825977,2501.387143,795.895909,8754.855,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,14995,NaN,NaN,92.753150,492.405318,53.746945,1847.551250,615.850417,7390.205,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14996,14996,NaN,NaN,5755.969848,3081.110590,249.962377,6776.757778,1793.847647,30495.410,17.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14997,14997,NaN,NaN,1611.702258,3602.147114,110.065038,5461.977500,1213.772778,21847.910,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14998,14998,NaN,NaN,1279.804147,5499.956792,123.578767,18935.013333,4733.753333,85207.560,14.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Predicting:

In [29]:
pd.Series(result.columns).value_counts()

prism_consumer_id                 1
DQ_TARGET                         1
amount_mean_EDUCATION             1
amount_mean_ENTERTAINMENT         1
amount_mean_ESSENTIAL_SERVICES    1
                                 ..
amount_sum_PAYCHECK               1
amount_sum_PENSION                1
amount_sum_PETS                   1
amount_sum_REFUND                 1
amount_mean_first_12_months       1
Name: count, Length: 256, dtype: int64

In [30]:
result = result.fillna(0)

In [31]:
cids = c_df.prism_consumer_id.unique()

X = cids
y = c_df["DQ_TARGET"]

X_2, test_cids, y_2, y_test = train_test_split(
    X, y, 
    test_size=0.2, random_state=16
)
train_cids, valid_cids, y_train, y_val = train_test_split(
    X_2, y_2, 
    test_size=0.25, random_state=16
)

# Features and labels for train, valid, and test sets:
X_train = result[result.prism_consumer_id.isin(train_cids)]
y_train = X_train.DQ_TARGET

X_valid = result[result.prism_consumer_id.isin(valid_cids)]
y_valid = X_valid.DQ_TARGET

X_test  = result[result.prism_consumer_id.isin(test_cids)]
y_test  = X_test.DQ_TARGET

X_train.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
X_valid.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
X_test.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)

len(X_train), len(X_valid), len(X_test)

C:\Users\bdion\AppData\Local\Temp\ipykernel_2628\1919337731.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
C:\Users\bdion\AppData\Local\Temp\ipykernel_2628\1919337731.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_valid.drop(columns=['prism_consumer_id', 'DQ_TARGET'], inplace=True)
C:\Users\bdion\AppData\Local\Temp\ipykernel_2628\1919337731.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/index

(9000, 3000, 3000)

In [32]:
clf = LogisticRegression(random_state=420, penalty=None).fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.93      1.00      0.96      2799
         1.0       0.00      0.00      0.00       201

    accuracy                           0.93      3000
   macro avg       0.47      0.50      0.48      3000
weighted avg       0.87      0.93      0.90      3000



c:\Users\bdion\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [33]:
metrics.roc_auc_score(y_test, y_pred)

0.4983922829581994

In [34]:
y_pred.sum()

9.0

In [35]:
y_test.sum()

201.0

In [36]:
y_pred = clf.predict(X_valid)
print(classification_report(y_valid, y_pred))

              precision    recall  f1-score   support

         0.0       0.93      1.00      0.96      2779
         1.0       0.14      0.01      0.02       221

    accuracy                           0.92      3000
   macro avg       0.53      0.50      0.49      3000
weighted avg       0.87      0.92      0.89      3000



In [37]:
metrics.roc_auc_score(y_valid, y_pred)

0.5023658368598359

In [38]:
coefficients = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': clf.coef_[0]})

coefficients.sort_values('Coefficient', ascending=False)[:50]

,Feature,Coefficient
112,amount_std_DEPOSIT,3.969240e-04
116,amount_std_EXTERNAL_TRANSFER,3.960232e-04
135,amount_std_PAYCHECK,3.724526e-04
102,amount_std_ATM_CASH,3.251619e-04
4,avg_monthly_outflow,3.064573e-04
143,amount_std_TAX,2.275395e-04
74,amount_sum_GIFTS_DONATIONS,1.804214e-04
87,amount_sum_OVERDRAFT,1.585134e-04
99,amount_sum_TRAVEL,1.391959e-04
119,amount_std_GAMBLING,1.086174e-04


- Positive coefficients → Increase probability of the positive class.
- Negative coefficients → Decrease probability.

In [39]:
clf = LogisticRegression(random_state=420, penalty=None, class_weight='balanced').fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.96      0.67      0.79      2799
         1.0       0.12      0.60      0.19       201

    accuracy                           0.67      3000
   macro avg       0.54      0.64      0.49      3000
weighted avg       0.90      0.67      0.75      3000



c:\Users\bdion\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [40]:
metrics.roc_auc_score(y_test, y_pred)

0.6361147104776226

In [41]:
y_pred.sum()

1044.0

In [42]:
y_test.sum()

201.0

In [43]:
y_pred = clf.predict(X_valid)
print(classification_report(y_valid, y_pred))

              precision    recall  f1-score   support

         0.0       0.95      0.68      0.79      2779
         1.0       0.12      0.58      0.20       221

    accuracy                           0.67      3000
   macro avg       0.54      0.63      0.50      3000
weighted avg       0.89      0.67      0.75      3000



In [44]:
metrics.roc_auc_score(y_valid, y_pred)

0.6274840879967566

In [45]:
coefficients = pd.DataFrame({'Feature': X_train.columns, 'Coefficient': clf.coef_[0]})

coefficients.sort_values('Coefficient', ascending=False)[:50]

,Feature,Coefficient
87,amount_sum_OVERDRAFT,0.000361
135,amount_std_PAYCHECK,0.000273
131,amount_std_MISCELLANEOUS,0.000223
120,amount_std_GENERAL_MERCHANDISE,0.000218
94,amount_sum_RTO_LTO,0.000209
102,amount_std_ATM_CASH,0.000201
74,amount_sum_GIFTS_DONATIONS,0.000192
240,amount_median_TRAVEL,0.000170
116,amount_std_EXTERNAL_TRANSFER,0.000159
159,amount_mean_DEPOSIT,0.000154
